# Giai đoạn 1: Data Collection (Cào dữ liệu TMDB)

Notebook này **sinh hai file CSV** trong thư mục làm việc hiện tại (thường là `Notebook_Report/`):

- `cinesense_movies.csv`
- `cinesense_reviews.csv`

**Toàn bộ logic cào nằm trong cell code bên dưới** — không `import` từ `etl_pipeline` hay file Python khác trong repo; chỉ dùng `httpx`, `pandas`, `beautifulsoup4` và thư viện chuẩn.

Luồng: `/genre/movie/list` → `/movie/popular` hoặc `/movie/top_rated` (phân trang) → `/movie/{id}/reviews` (tối đa `MAX_REVIEWS_PER_MOVIE`, số trang = `(n//20)+1`). Lọc review: heuristic nhiễu + (tuỳ chọn) chỉ giữ `en`/`unknown` theo tỉ lệ ký tự ASCII.

**Schema CSV** khớp notebook 02 / seed (`tmdb_id`, `title`, `overview`, … cho movies; `review_id`, `tmdb_id`, `content`, … cho reviews).

## Tham số

- `MAX_MOVIES` (mặc định 5000), `MOVIE_SOURCE`, `MAX_REVIEWS_PER_MOVIE` (mặc định 20), `ONLY_ENGLISH_REVIEWS`.
- `TMDB_API_KEY` trong `.env` ở **root repo** hoặc biến môi trường.

> Chạy với working directory là `Notebook_Report/` nếu muốn ghi CSV đúng chỗ quen thuộc.



In [1]:
"""Notebook 1 — cào TMDB và ghi CSV. Toàn bộ logic nằm trong cell này (không import mã nguồn dự án)."""
from __future__ import annotations

import math
import os
import re
import time
import unicodedata
from datetime import date
from pathlib import Path

import httpx
import pandas as pd
from bs4 import BeautifulSoup

# --- .env (chỉ để đọc TMDB_API_KEY; không nạp package nội bộ repo) ---
_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "Notebook_Report" else _here
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
except ImportError:
    pass

TMDB_API_KEY = os.getenv("TMDB_API_KEY", "").strip()
TMDB_BASE = "https://api.themoviedb.org/3"
TMDB_LANG = "en-US"
REQUEST_DELAY_S = 0.12
TIMEOUT_S = 30.0

# ---------------------------------------------------------------------------
# Tham số crawl
# ---------------------------------------------------------------------------
MAX_MOVIES = 5000
MOVIE_SOURCE = "popular"  # "popular" | "top_rated"
MAX_REVIEWS_PER_MOVIE = 20
ONLY_ENGLISH_REVIEWS = True

OUT_MOVIES = "cinesense_movies.csv"
OUT_REVIEWS = "cinesense_reviews.csv"

MOVIE_PAGE_SIZE = 20
PAGES_END = math.ceil(MAX_MOVIES / MOVIE_PAGE_SIZE)
review_max_pages = (MAX_REVIEWS_PER_MOVIE // 20) + 1

if not TMDB_API_KEY:
    raise ValueError(
        "Thiếu TMDB_API_KEY. Đặt trong .env ở root repo hoặc export biến môi trường."
    )

# ---------------------------------------------------------------------------
# Heuristic lọc review (tương đương pipeline ETL — code độc lập trong notebook)
# ---------------------------------------------------------------------------
_URL_PATTERN = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
_EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b", flags=re.IGNORECASE)
_MENTION_HASHTAG_PATTERN = re.compile(r"(?:(?<!\w)[@#]\w+)", flags=re.UNICODE)
_EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001F5FF\U0001F600-\U0001F64F\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF"
    "\u2600-\u26FF\u2700-\u27BF]+",
    flags=re.UNICODE,
)


def _normalize_unicode(text: str) -> str:
    if not text:
        return ""
    return unicodedata.normalize("NFC", text)


def _clean_html(text: str) -> str:
    if not text:
        return ""
    soup = BeautifulSoup(text, "html.parser")
    for element in soup(["script", "style"]):
        element.decompose()
    return soup.get_text(separator=" ")


def _remove_urls_emails_handles(text: str) -> str:
    if not text:
        return ""
    text = _URL_PATTERN.sub(" ", text)
    text = _EMAIL_PATTERN.sub(" ", text)
    text = _MENTION_HASHTAG_PATTERN.sub(" ", text)
    return text


def _remove_emojis(text: str) -> str:
    if not text:
        return ""
    return _EMOJI_PATTERN.sub(" ", text)


def _normalize_whitespace(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"[\n\t\r]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _preprocess_for_noise_check(text: str) -> str:
    if not text:
        return ""
    t = _normalize_unicode(text)
    t = _clean_html(t)
    t = _remove_urls_emails_handles(t)
    t = _remove_emojis(t)
    t = _normalize_whitespace(t)
    return t.lower()


def is_noisy_review(
    text: str,
    min_chars: int = 20,
    min_alpha_ratio: float = 0.3,
    min_alpha_tokens: int = 3,
) -> bool:
    if not text:
        return True
    raw = (text or "").strip()
    if len(raw) < min_chars:
        return True
    cleaned = _preprocess_for_noise_check(raw)
    if not cleaned:
        return True
    total_len = len(cleaned)
    alpha_count = sum(ch.isalpha() for ch in cleaned)
    if total_len == 0:
        return True
    if alpha_count / total_len < min_alpha_ratio:
        return True
    tokens = [tok for tok in cleaned.split() if tok.isalpha()]
    if len(tokens) < min_alpha_tokens:
        return True
    return False


def infer_review_language(content: str) -> str:
    text = (content or "").strip()
    if len(text) < 20:
        return "unknown"
    ascii_letters = sum(ch.isascii() and ch.isalpha() for ch in text)
    alpha_total = sum(ch.isalpha() for ch in text)
    if alpha_total == 0:
        return "unknown"
    ratio = ascii_letters / alpha_total
    return "en" if ratio >= 0.85 else "unknown"


# ---------------------------------------------------------------------------
# TMDB HTTP (httpx)
# ---------------------------------------------------------------------------
def tmdb_request(
    client: httpx.Client,
    endpoint: str,
    extra_params: dict | None = None,
) -> dict:
    params = {"api_key": TMDB_API_KEY}
    if extra_params:
        params.update(extra_params)
    time.sleep(REQUEST_DELAY_S)
    r = client.get(endpoint, params=params)
    if r.status_code == 429:
        wait = int(r.headers.get("Retry-After", "3"))
        print(f"429 — chờ {wait}s …")
        time.sleep(wait)
        time.sleep(REQUEST_DELAY_S)
        r = client.get(endpoint, params=params)
    r.raise_for_status()
    return r.json()


def load_genre_map(client: httpx.Client) -> dict[int, str]:
    data = tmdb_request(client, "/genre/movie/list", {"language": TMDB_LANG})
    out: dict[int, str] = {}
    for g in data.get("genres") or []:
        if "id" in g and "name" in g:
            out[int(g["id"])] = str(g["name"])
    return out


def parse_movie(item: dict, genre_map: dict[int, str]) -> dict:
    mid = int(item["id"])
    gids = item.get("genre_ids") or []
    names = [genre_map.get(int(g), "Unknown") for g in gids]
    genres_str = ", ".join(sorted({n for n in names if n}))
    rd = item.get("release_date") or ""
    release_date = ""
    if rd:
        try:
            release_date = date.fromisoformat(str(rd)[:10]).isoformat()
        except ValueError:
            release_date = str(rd)[:10]
    return {
        "tmdb_id": mid,
        "title": item.get("title") or item.get("original_title") or "",
        "overview": item.get("overview") or "",
        "genres": genres_str,
        "release_date": release_date,
        "poster_path": item.get("poster_path") or "",
        "vote_average": item.get("vote_average", ""),
        "vote_count": item.get("vote_count", ""),
        "popularity": item.get("popularity", ""),
    }


def parse_review(r: dict, movie_tmdb_id: int) -> dict:
    ad = r.get("author_details") or {}
    rating = ad.get("rating", "")
    if rating is None:
        rating = ""
    return {
        "review_id": r.get("id") or "",
        "tmdb_id": movie_tmdb_id,
        "author": r.get("author") or "",
        "author_name": ad.get("name") or r.get("author") or "",
        "content": r.get("content") or "",
        "rating": rating,
        "avatar_path": ad.get("avatar_path") or "",
        "created_at": r.get("created_at") or "",
        "url": r.get("url") or "",
        "source": "tmdb",
    }


def fetch_movie_page(client: httpx.Client, page: int) -> list[dict]:
    ep = "/movie/popular" if MOVIE_SOURCE == "popular" else "/movie/top_rated"
    data = tmdb_request(
        client,
        ep,
        {"language": TMDB_LANG, "page": page},
    )
    return list(data.get("results") or [])


def fetch_reviews_for_movie(client: httpx.Client, movie_id: int) -> list[dict]:
    all_r: list[dict] = []
    for rp in range(1, review_max_pages + 1):
        data = tmdb_request(client, f"/movie/{movie_id}/reviews", {"page": rp})
        batch = list(data.get("results") or [])
        if not batch:
            break
        all_r.extend(batch)
        total_pages = int(data.get("total_pages") or 1)
        if rp >= total_pages:
            break
    return all_r[:MAX_REVIEWS_PER_MOVIE]


# ---------------------------------------------------------------------------
# Crawl chính
# ---------------------------------------------------------------------------
movies_rows: list[dict] = []
reviews_rows: list[dict] = []
seen_tmdb: set[int] = set()

with httpx.Client(base_url=TMDB_BASE, timeout=TIMEOUT_S) as client:
    genre_map = load_genre_map(client)
    print(f"Đã tải {len(genre_map)} thể loại.")

    for page in range(1, PAGES_END + 1):
        if len(movies_rows) >= MAX_MOVIES:
            break
        items = fetch_movie_page(client, page)
        for item in items:
            if len(movies_rows) >= MAX_MOVIES:
                break
            if not item.get("id"):
                continue
            mid = int(item["id"])
            if mid in seen_tmdb:
                continue
            seen_tmdb.add(mid)
            row = parse_movie(item, genre_map)
            movies_rows.append(row)
            raw_reviews = fetch_reviews_for_movie(client, mid)
            for r in raw_reviews:
                pr = parse_review(r, mid)
                if not pr["content"] or is_noisy_review(pr["content"]):
                    continue
                lang = infer_review_language(pr["content"])
                if ONLY_ENGLISH_REVIEWS and lang not in ("en", "unknown"):
                    continue
                reviews_rows.append(pr)
        print(f"Trang {page}/{PAGES_END} | phim={len(movies_rows)} | review={len(reviews_rows)}")

movies_df = pd.DataFrame(movies_rows)
reviews_df = pd.DataFrame(reviews_rows)
movies_df.to_csv(OUT_MOVIES, index=False, encoding="utf-8", na_rep="")
reviews_df.to_csv(OUT_REVIEWS, index=False, encoding="utf-8", na_rep="")
print(f"Đã lưu: {OUT_MOVIES} ({len(movies_df)} dòng), {OUT_REVIEWS} ({len(reviews_df)} dòng)")
display(movies_df.head(2))
display(reviews_df.head(2))



Đã tải 19 thể loại.
Trang 1/250 | phim=20 | review=71
Trang 2/250 | phim=40 | review=132
Trang 3/250 | phim=60 | review=203
Trang 4/250 | phim=80 | review=233
Trang 5/250 | phim=100 | review=305
Trang 6/250 | phim=120 | review=327
Trang 7/250 | phim=140 | review=410
Trang 8/250 | phim=160 | review=517
Trang 9/250 | phim=180 | review=646
Trang 10/250 | phim=200 | review=751
Trang 11/250 | phim=220 | review=850
Trang 12/250 | phim=240 | review=894
Trang 13/250 | phim=260 | review=978
Trang 14/250 | phim=280 | review=1057
Trang 15/250 | phim=300 | review=1141
Trang 16/250 | phim=320 | review=1203
Trang 17/250 | phim=340 | review=1301
Trang 18/250 | phim=360 | review=1416
Trang 19/250 | phim=380 | review=1494
Trang 20/250 | phim=400 | review=1564
Trang 21/250 | phim=420 | review=1638
Trang 22/250 | phim=440 | review=1698
Trang 23/250 | phim=460 | review=1794
Trang 24/250 | phim=480 | review=1876
Trang 25/250 | phim=500 | review=1947
Trang 26/250 | phim=520 | review=2057
Trang 27/250 | phim

,tmdb_id,title,overview,genres,release_date,poster_path,vote_average,vote_count,popularity
0,1523145,Your Heart Will Be Broken,High school student Polina is saved from bully...,"Drama, Romance",2026-03-26,/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg,5.556,9,718.2018
1,875828,Peaky Blinders: The Immortal Man,After his estranged son gets embroiled in a Na...,"Crime, Drama",2026-03-05,/gRMalasZEzsZi4w2VFuYusfSfqf.jpg,7.363,467,290.8743


,review_id,tmdb_id,author,author_name,content,rating,avatar_path,created_at,url,source
0,69b2a90fab3233b77a685bd7,875828,CinemaSerf,CinemaSerf,Anyone remember Michael Elphick’s “Private Sch...,7.0,/9HcQx0Yfbxy8eqr3ft66X9uWMf0.jpg,2026-03-12T11:52:47.670Z,https://www.themoviedb.org/review/69b2a90fab32...,tmdb
1,6943d8d0a56edf751f705664,83533,Manuel São Bento,Manuel São Bento,FULL SPOILER-FREE REVIEW @ https://movieswetex...,5.0,,2025-12-18T10:34:56.770Z,https://www.themoviedb.org/review/6943d8d0a56e...,tmdb
